In [1]:
# Cell 2: Mount Google Drive and set up Git user
import os
from google.colab import drive
drive.mount('/content/drive')

# Configure Git (replace with your own name/email)
!git config --global user.name "Your Name"
!git config --global user.email "your.email@example.com"

# Clone the repository (change URL if needed)
!git clone https://github.com/inshiright/chestxray-classification.git
%cd chestxray-classification

print(f"Current working directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'chestxray-classification' already exists and is not an empty directory.
/content/chestxray-classification
Current working directory: /content/chestxray-classification


In [2]:
# Cell 3: Install Python dependencies
!pip install -r requirements.txt
!pip install transformers shap lime pytorch-gradcam

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0

  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
^C


In [ ]:

import shutil

# 2. Define source and destination
SOURCE_ZIP = "/content/drive/MyDrive/ADL_Data/data.zip"

DESTINATION_DIR = "/content/dataset"

# 3. Create the local directory
os.makedirs(DESTINATION_DIR, exist_ok=True)

print("Starting Transfer from Drive to Local SSD (this is fast)...")

# 4. Copy the zip file
!cp "{SOURCE_ZIP}" /content/data_local.zip
# !gdown "1bO0S2WeRnu1l25wDbhFmgrP9yUK_g93J" -O /content/data_local.zip

print("Transfer complete. Starting extraction (this may take 15-20 mins)...")

# 5. Unzip the file into the local folder
!unzip -q /content/data_local.zip -d "{DESTINATION_DIR}"

# 6. Clean up the local zip
os.remove("/content/data_local.zip")

print(f"Done! Your images are ready at {DESTINATION_DIR}")

In [ ]:
# Cell 5: Update config.py to point to the dataset location
# (The repository's config.py may exist, but we override it to ensure correct paths)
%%writefile src/config.py
import os

ROOT_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))

# Pointing to the extracted dataset location
DATASET_DIR = "/content/dataset"
CSV_PATH = os.path.join(DATASET_DIR, "Data_Entry_2017.csv")

# Hyperparameters
IMAGE_SIZE = 224
NUM_CLASSES = 14
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-4

# Active Model
MODEL_NAME = "raddino"

# Checkpoint Settings
CHECKPOINT_DIR = os.path.join(ROOT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
RESUME_CHECKPOINT_PATH = None

In [ ]:

!python src/main.py

# To resume training from a checkpoint (if available):
# !python src/resume_train.py checkpoints/raddino_best_model.pth